In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datasets/creditcard.csv")

In [2]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

FEATURES = [c for c in df.columns if c not in ("Class",)]

def fit_eval(train, test, use_smote):
    steps = [("sc", StandardScaler())]
    if use_smote:
        steps.append(("sm", SMOTE(random_state=42)))
    steps.append(("clf", LogisticRegression(max_iter=2000,
                  class_weight=None if use_smote else "balanced")))
    pipe = ImbPipeline(steps).fit(train[FEATURES], train["Class"])
    p = pipe.predict_proba(test[FEATURES])[:, 1]
    return {"pr_auc": average_precision_score(test["Class"], p),
            "roc_auc": roc_auc_score(test["Class"], p)}